# 04 - Export para Clinical Eval App

Este notebook prepara el paquete de evaluacion para `clinical_eval_app/`:
- Lee `results.jsonl` de un escenario (A/B/C/D/E/F).
- Filtra casos correctos (`class_match=True`) si se desea.
- Ordena de mejor a peor por `iou_score + proximity_score`.
- Selecciona casos con normalizacion por clase (`AD/HP/ASS`) para evitar sesgo de tipo.
- Genera imagenes con **solo Ground Truth bbox** dibujado en verde.
- Exporta `clinical_eval_app/data/cases.json` y metadatos auxiliares.

Compatibilidad Scenario F:
- Soporta payload `AssistedClinicalReport` (campo `diagnostic_rationale`).
- Si no hay campo unico de justificacion, compone texto desde morfologia + patron vascular.

In [7]:
from __future__ import annotations

import json
import sys
from pathlib import Path

base = Path.cwd().resolve()
project_root = next(
    (candidate for candidate in [base, *base.parents] if (candidate / 'pyproject.toml').exists()),
    base,
 )
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preprocessing.export_clinical_eval_data import (
    ExportConfig,
    prepare_and_select_cases,
    resolve_paths,
    export_selected_cases,
    read_jsonl_records,
)

print('OK: imports cargados')

OK: imports cargados


## Paso 1 - Configuracion del export
En la siguiente celda defines escenario, run, cantidad de casos y politica de reemplazo.

In [8]:
# =========================
# Configuracion del export
# =========================
SCENARIO_NAME = 'scenario_F'       # scenario_A .. scenario_F
RUN_ID = 'run_20260429_141632'         # ejemplo: 'run_20260407_103355' o '20260407_103355' (None => ultimo run)
RUN_PATH_OVERRIDE = None           # Path absoluto/relativo a run_... si quieres forzar

TOP_K = 50                         # cuantas imagenes exportar
ONLY_CLASS_MATCH = True            # solo muestras correctas
REQUIRE_STATUS_OK = True           # exige status == 'ok'
DROP_EMPTY_JUSTIFICATION = True    # descarta si falta clinical_justification

# Seleccion balanceada por clase (normalizacion por tipo)
NORMALIZE_BY_CLASS = True
NORMALIZATION_MODE = 'proportional'  # 'proportional' (como scenario/manifest) o 'uniform'
VALID_CLASSES = ('AD', 'HP', 'ASS')

# Modo de export a la app:
# - 'replace': escribe en data/ (raiz) y limpia solo data/images/
# - 'scenario_id': escribe en data/<scenario_uid>/ con carpeta corta (ej: 20260423_171622F)
# - 'append': agrega al data/ (raiz), sin duplicar imagenes ya exportadas
EXPORT_MODE = 'scenario_id'
SCENARIO_UID_OVERRIDE = None  # opcional; si None se usa formato corto RUNID+letraEscenario

# Score de ranking solicitado: iou + proximity
# (si falta alguno, se trata como 0.0)

# Salida para la web app
OUTPUT_APP_DIR = Path('clinical_eval_app')
OUTPUT_DATA_DIR = OUTPUT_APP_DIR / 'data'
OUTPUT_IMAGES_DIR = OUTPUT_DATA_DIR / 'images'
OUTPUT_CASES_JSON = OUTPUT_DATA_DIR / 'cases.json'
OUTPUT_EXPORT_CSV = OUTPUT_DATA_DIR / 'cases_export_debug.csv'

# Dibujo
GT_COLOR_BGR = (0, 200, 0)
GT_THICKNESS = 3
SCALE_MAX = 1000

print(f'Configuracion lista. EXPORT_MODE={EXPORT_MODE}')

Configuracion lista. EXPORT_MODE=scenario_id


## Paso 2 - Resolucion de rutas y utilidades
La celda siguiente prepara funciones auxiliares para localizar el proyecto, resolver el run seleccionado y convertir bboxes.
Tambien deja definida la ruta final de `results.jsonl` a procesar.

In [9]:
config = ExportConfig(
    scenario_name=SCENARIO_NAME,
    run_id=RUN_ID,
    run_path_override=RUN_PATH_OVERRIDE,
    top_k=max(0, int(TOP_K)),
    only_class_match=bool(ONLY_CLASS_MATCH),
    require_status_ok=bool(REQUIRE_STATUS_OK),
    drop_empty_justification=bool(DROP_EMPTY_JUSTIFICATION),
    normalize_by_class=bool(NORMALIZE_BY_CLASS),
    normalization_mode=str(NORMALIZATION_MODE),
    valid_classes=tuple(VALID_CLASSES),
    export_mode=str(EXPORT_MODE),
    scenario_uid_override=SCENARIO_UID_OVERRIDE,
    output_app_dir=OUTPUT_APP_DIR,
    gt_color_bgr=GT_COLOR_BGR,
    gt_thickness=int(GT_THICKNESS),
    scale_max=int(SCALE_MAX),
)

paths = resolve_paths(config)
RESULTS_JSONL = paths.results_jsonl
PROJECT_ROOT = paths.project_root
RUN_DIR = paths.run_dir
OUTPUT_APP_DIR = paths.output_app_dir
OUTPUT_DATA_DIR = paths.output_data_dir
OUTPUT_IMAGES_DIR = paths.output_images_dir
OUTPUT_CASES_JSON = paths.output_cases_json
OUTPUT_EXPORT_CSV = paths.output_export_csv

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'RUN_DIR:      {RUN_DIR}')
print(f'JSONL:        {RESULTS_JSONL}')
print(f'APP_DIR:      {OUTPUT_APP_DIR}')
print(f'DATA_DIR:     {OUTPUT_DATA_DIR}')

PROJECT_ROOT: C:\Users\david\Desktop\TFG\TFG_VLM_Medical
RUN_DIR:      C:\Users\david\Desktop\TFG\TFG_VLM_Medical\data\processed\scenario_results\scenario_F\run_20260429_141632
JSONL:        C:\Users\david\Desktop\TFG\TFG_VLM_Medical\data\processed\scenario_results\scenario_F\run_20260429_141632\results.jsonl
APP_DIR:      C:\Users\david\Desktop\TFG\TFG_VLM_Medical\clinical_eval_app
DATA_DIR:     C:\Users\david\Desktop\TFG\TFG_VLM_Medical\clinical_eval_app\data


## Paso 3 - Filtro, ranking y normalizacion por tipo
Aqui se aplica el filtrado (status, class_match, justificacion, etc.) y se calcula score = `iou_score + proximity_score`.
Despues se selecciona el TOP_K con normalizacion por clase (`AD/HP/ASS`) para evitar sesgo por tipo y, dentro de cada tipo, priorizar los mejores scores.

In [10]:
selection = prepare_and_select_cases(config=config, paths=paths)
prepared = selection.prepared
selected = selection.selected
dist_prepared = selection.dist_prepared
dist_selected = selection.dist_selected

print(f'Registros leidos: {len(read_jsonl_records(RESULTS_JSONL))}')
print(f'Candidatos tras filtros: {len(prepared)}')
print(f'Seleccion final (TOP_K): {len(selected)}')

if dist_prepared:
    print('Distribucion candidatos (normalizada):', dist_prepared)
if dist_selected:
    print('Distribucion seleccion final:', dist_selected)
if selection.skipped_by_reason:
    print('Descartes por motivo:', selection.skipped_by_reason)

for i, row in enumerate(selected[:5], start=1):
    print(
        f"{i:02d}. image_id={row['image_id']} score={row['quality_score']:.4f} "
        f"iou={row['iou_score']} prox={row['proximity_score']} cls={row['ground_truth_cls_norm']}"
    )

Registros leidos: 501
Candidatos tras filtros: 500
Seleccion final (TOP_K): 50
Distribucion candidatos (normalizada): {'HP': 79, 'AD': 340, 'ASS': 81}
Distribucion seleccion final: {'AD': 33, 'ASS': 9, 'HP': 8}
Descartes por motivo: {'status_not_ok': 1}
01. image_id=454 score=1.9146 iou=0.939 prox=0.9755997750000422 cls=AD
02. image_id=780 score=1.8786 iou=0.9164325111874284 prox=0.962150083952502 cls=AD
03. image_id=462 score=1.8215 iou=0.875 prox=0.9464946545954827 cls=AD
04. image_id=262 score=1.8186 iou=0.8725225822613206 prox=0.94604266213887 cls=AD
05. image_id=458 score=1.8181 iou=0.8774891067799786 prox=0.9406488184398978 cls=AD


## Paso 4 - Export de datos para la app (3 modos)
Esta celda exporta en uno de estos modos:
- `replace`: escribe en `clinical_eval_app/data/` y limpia solo `data/images` (no toca subcarpetas de escenarios).
- `scenario_id`: crea una carpeta dedicada `clinical_eval_app/data/<scenario_uid>/` con `cases.json`, `cases_export_debug.csv` e `images/`.
  - Si `SCENARIO_UID_OVERRIDE=None`, el folder se genera corto como `RUNID + letraEscenario` (ejemplo: `20260423_171622F`).
  - Las imagenes se guardan con nombre corto y estable (`img_<id>.jpg`).
- `append`: trabaja sobre `clinical_eval_app/data/` (raiz), agrega/actualiza casos y evita duplicar imagenes ya existentes en `data/images/`.

In [11]:
export_result = export_selected_cases(config=config, paths=paths, selected=selected)
cases = export_result.cases

print(f'Export completado. Cases: {len(cases)}')
print(f'JSON: {export_result.output_cases_json.resolve()}')
print(f'CSV:  {export_result.output_export_csv.resolve()}')
print(f'IMG:  {export_result.output_images_dir.resolve()}')
print(f'Modo export: {export_result.mode}')
print(f'Scenario UID: {export_result.scenario_uid}')
print(f'Casos nuevos generados en esta corrida: {export_result.new_cases_count}')
if export_result.mode == 'append':
    print(f'Casos existentes leidos: {export_result.existing_cases_count}')

Export completado. Cases: 50
JSON: C:\Users\david\Desktop\TFG\TFG_VLM_Medical\clinical_eval_app\data\20260429_141632F\cases.json
CSV:  C:\Users\david\Desktop\TFG\TFG_VLM_Medical\clinical_eval_app\data\20260429_141632F\cases_export_debug.csv
IMG:  C:\Users\david\Desktop\TFG\TFG_VLM_Medical\clinical_eval_app\data\20260429_141632F\images
Modo export: scenario_id
Scenario UID: 20260429_141632F
Casos nuevos generados en esta corrida: 50


## Paso 5 - Verificacion rapida
Se imprime una muestra de los primeros casos para comprobar formato y rutas resultantes.
Si el total de casos es 0, revisa filtros como `ONLY_CLASS_MATCH` o la disponibilidad de justificaciones.

In [12]:
# Vista rapida de salida
print('Primeros 3 casos exportados:')
for item in cases[:3]:
    print(json.dumps(item, ensure_ascii=False, indent=2))

print('...')
print(f'Total casos exportados: {len(cases)}')

Primeros 3 casos exportados:
{
  "id_imagen": "20260429_141632f__google_gemma_4_e4b_q8_0__454",
  "image_path": "data/20260429_141632F/images/img_454.jpg",
  "ground_truth_class": "AD",
  "id_modelo": "google/gemma-4-e4b@q8_0",
  "clinical_justification": "La lesión presenta una morfología de tipo 'S' (sessile) o subsésil, caracterizada por su base ancha e integración con la mucosa circundante. La superficie es marcadamente lobulada y nodular, lo que sugiere un crecimiento exofítico complejo. El patrón vascular superficial exhibe ramificaciones finas y densas, típico de una proliferación adenomatosa. Estos hallazgos morfológicos son altamente sugestivos de un Adenoma (AD), justificando el diagnóstico histológico confirmado.",
  "scenario_uid": "20260429_141632F",
  "source_image_id": "454"
}
{
  "id_imagen": "20260429_141632f__google_gemma_4_e4b_q8_0__780",
  "image_path": "data/20260429_141632F/images/img_780.jpg",
  "ground_truth_class": "AD",
  "id_modelo": "google/gemma-4-e4b@q8_0"